# Spark Structured Streaming App

![Alt Text](/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/visuals/spark_workflow_diagram.png)

Configure the local system environment properties to install the required coordinates for the Spark SQL Kafka streaming architecture.
Import the pymongo and pyspark libraries. pymongo is for the MongoClient class and pyspark is for SparkSession and the SQL functions and schema data types.

In [1]:
import findspark
findspark.init() #help to resolve the pyspark module not found issue due to nbconvert and running in container environment
from pyspark.sql.functions import udf
from pyspark.sql.types import TimestampType

# UDF breaks Spark's column lineage, fully detaching watermark metadata and resolving
# "More than one event time columns are available." error when running spark for ab_join and abc_join
strip_watermark = udf(lambda ts: ts, TimestampType())

import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

Determine names for each of the three topics, one for each camera, and making sure that the topic names match those from producer_utils.ipynb.
Configure the host IP as well.

In [2]:
# Determine the topic names for each set of camera events A, B and C
topic_A_name = "camera_event_a"
topic_B_name = "camera_event_b"
topic_C_name = "camera_event_c"
# Configure Host IP
hostip = "kafka"

Initialise the local SparkSession engine

In [3]:
# Create the spark engine
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('Spark Multi-Streaming App for AWAS')
    .getOrCreate()
)
# Configure the log threshold to suppress noise and only output a message in the event of a runtime error
spark.sparkContext.setLogLevel("ERROR")

Construct the explicit structural StructType data schema to parse incoming JSON payloads. The schema maps out the exact column names (eg: camera_id, batch_id, car_plate, etc) and assigns their corresponding strict scalar types (eg: IntegerType, StringType, etc) to enforce types across streaming pipelines.

In [4]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", TimestampType()),
    StructField("speed_reading", DoubleType())
])

For each Kafka broker endpoint (one per camera), set up individual topic streams.

In [5]:
# Topic stream A
topic_stream_a = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', f'{hostip}:29092')
    .option('subscribe', topic_A_name)
    .load()
)

# Topic stream B
topic_stream_b = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', f'{hostip}:29092')
    .option('subscribe', topic_B_name)
    .load()
)

# Topic stream C
topic_stream_c = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', f'{hostip}:29092')
    .option('subscribe', topic_C_name)
    .load()
)

print(topic_stream_a)

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]


Form output streams from each topic stream by extracting, decoding and transforming the raw binary messages from Kafka topics. Watermarks (with 5-minute intervals) are also immediately applied directly onto the dataframes after unpacking the JSON strings.

In [6]:
# Output stream A
output_stream_a = (
    topic_stream_a
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            ArrayType(event_schema)
        )
        .alias('data')
    )
    .select(explode(col("data")).alias("event"))
    .select("event.*")
    .withWatermark('timestamp', '5 minutes')
)

# Output stream B
output_stream_b = (
    topic_stream_b
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            ArrayType(event_schema)
        )
        .alias('data')
    )
    .select(explode(col("data")).alias("event"))
    .select("event.*")
    .withWatermark('timestamp', '5 minutes')
)

# Output stream C
output_stream_c = (
    topic_stream_c
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            ArrayType(event_schema)
        )
        .alias('data')
    )
    .select(explode(col("data")).alias("event"))
    .select("event.*")
    .withWatermark('timestamp', '5 minutes')
)

print(output_stream_a)

DataFrame[event_id: string, batch_id: int, car_plate: string, camera_id: int, timestamp: timestamp, speed_reading: double]


Perform time-windowed left outer stream join to merge output streams A and B. The join ensures that the car_plate matches between both streams and that the timestamps are at most 2 minutes apart (determined by the watermark)

In [7]:
ab_join = (
    output_stream_a.alias("a")
    .join(
        output_stream_b.alias("b"),
        expr("""
            a.car_plate = b.car_plate AND
            b.timestamp >= a.timestamp AND
            b.timestamp <= a.timestamp + interval 2 minutes
        """),
        "leftOuter"
    )
    .select(
        col("a.event_id").alias("a_event_id"),
        col("a.car_plate").alias("car_plate"),
        col("a.timestamp").alias("a_timestamp"),

        col("b.event_id").alias("b_event_id"),
        strip_watermark(col("b.timestamp")).alias("b_timestamp")
    )
)

Perform time-windowed left outer stream join again, this time merging the merged stream from the code cell above with output stream C. The join ensures that the car_plate matches between both streams. A secondary event-time constraint is applied here and verifies that camera C's timestamp occurs within the 2-minute interval after camera B's event to ensure a chronologically valid vehicle tracking path across all three cameras.

In [8]:
abc_join = (
    ab_join.alias("ab")
    .join(
        output_stream_c.alias("c"),
        expr("""
            ab.car_plate = c.car_plate AND
            c.timestamp >= ab.a_timestamp AND
            c.timestamp <= ab.a_timestamp + interval 4 minutes
        """),
        "leftOuter"
    )
    .select(
        col("ab.car_plate").alias("car_plate"),

        col("ab.a_event_id"),
        col("ab.a_timestamp"),
        col("ab.b_event_id"),
        col("ab.b_timestamp"),

        col("c.event_id").alias("c_event_id"),
        strip_watermark(col("c.timestamp")).alias("c_timestamp"),
        col("c.speed_reading").alias("c_speed_reading")
    )
)

Create a dataset to combine the three streams and include only speed violations (calculated by 1km * 3600 seconds/hour / (end timestamp - start timestamp)) between cameras A and B and between cameras B and C. All three cameras are situated 1km apart from each other.

![Alt Text](/home/student/A2/FIT3182_A2/A2/34900403_33524815_assignment02/visuals/complete_journeys_workflow_diagram.png)

In [9]:
# Distances between cameras are 1km apart
DISTANCE_AB = 1.0 
DISTANCE_BC = 1.0 

# Join cameras A and B

ab_violations = (
    ab_join

    # Calculate travel time between cameras A and B in seconds
    .withColumn(
        "travel_time_seconds",
        unix_timestamp("b_timestamp") -
        unix_timestamp("a_timestamp")
    )

    # Calculate average speed
    .withColumn(
        "average_speed",
        lit(DISTANCE_AB) /
        (col("travel_time_seconds") / 3600.0)
    )

    # Filter speeding vehicles (camera B's speed limit is 110km/h)
    .filter(col("average_speed") > 110)

    # Match the MongoDB violations schema
    .select(
        expr("uuid()").alias("violation_id"),

        col("car_plate"),
        
        # Since the camera A events table has camera_id all be 1,
        # use "1" for the entry camera ID
        lit(1).alias("camera_id_start"),
        # Since the camera B events table has camera_id all be 2,
        # use "2" for the exit camera ID
        lit(2).alias("camera_id_end"),

        col("a_timestamp").alias("timestamp_start"),
        col("b_timestamp").alias("timestamp_end"),

        round(col("average_speed"), 2)
            .alias("speed_reading")
    )
)


# Join cameras B and C 

bc_violations = (
    abc_join

    .filter(col("b_event_id").isNotNull())

    # Calculate travel time between cameras B and C in seconds
    .withColumn(
        "travel_time_seconds",
        unix_timestamp("c_timestamp") -
        unix_timestamp("b_timestamp")
    )

    # Calculate average speed
    .withColumn(
        "average_speed",
        lit(DISTANCE_BC) /
        (col("travel_time_seconds") / 3600.0)
    )

    # Filter speeding vehicles (camera C's speed limit is 90km/h)
    .filter(col("average_speed") > 90)

    # Match the MongoDB violations schema
    .select(
        expr("uuid()").alias("violation_id"),

        col("car_plate"),

        # Since the camera B events table has camera_id all be 2,
        # use "2" for the entry camera ID
        lit(2).alias("camera_id_start"),
        # Since the camera C events table has camera_id all be 3,
        # use "3" for the exit camera ID
        lit(3).alias("camera_id_end"),

        col("b_timestamp").alias("timestamp_start"),
        col("c_timestamp").alias("timestamp_end"),

        round(col("average_speed"), 2).alias("speed_reading")
    )
)

# Create a master violation dataset by unioning the two stream segments together
violations_df = (
    ab_violations
    .unionByName(bc_violations)
)

# Remove duplicate violation records by car plate and start/end timestamps
violations_df = violations_df.dropDuplicates([
    "car_plate",
    "timestamp_start",
    "timestamp_end"
])

Fliter the combined multi-stream join dataset to isolate completed vehicle journeys (by only taking records with a camera C event).

In [10]:
completed_journeys = (
    abc_join
    .filter(
        col("c_event_id").isNotNull()
    )
)

Fliter the combined multi-stream join dataset to isolate incomplete vehicle journeys (by only taking records with no camera C event).

In [11]:
incomplete_journeys = (
    abc_join
    .filter(
        col("c_event_id").isNull()
    )
)

Database writer (DbWriter class) to handle resilient, retry-protected network connections to MongoDB across separate partitions. The writer processes streaming violation rows in sequential order, executing atomic multi-conditional upserts to ensure that infraction metrics can be determined for each group by both car plate and day.

In [12]:
# Import time and pymongo errors
from pymongo.errors import ConnectionFailure, AutoReconnect
import time

# The database writer class
class DbWriter:

    # Initialise the object, specifying the intended names of the 
    # MongoDB database and collection to place the documents 
    def __init__(self, db_name, collection_name):
        self.db_name = db_name
        self.collection_name = collection_name

    # Called at the start of processing each partition in each output micro-batch
    def open(self, partition_id, epoch_id):
        retries = 3
        while retries > 0:
            try:
                self.mongo_client = MongoClient(
                    host='mongodb',
                    port=27017,
                    serverSelectionTimeoutMS=5000
                )
                self.db = self.mongo_client[self.db_name]
                return True
            except (ConnectionFailure, AutoReconnect) as e:
                retries -= 1
                if retries == 0:
                    raise e
                time.sleep(2)

    # Called once per row of the result dataframe
    def process(self, row):

        # Convert row in the data partition to a dictionary
        row_dict = row.asDict()

        car_plate = row_dict["car_plate"]
        ts_start = row_dict["timestamp_start"]

        # Obtains the violation day from the timestamp in the
        # format: "YYYY-MM-DD"
        violation_day = ts_start.strftime("%Y-%m-%d")

        # Constructs the sub-document for each document, 
        # specifically for a single speed violation
        event_entry = {
            "violation_id": row_dict["violation_id"],
            "camera_id_start": row_dict["camera_id_start"],
            "camera_id_end": row_dict["camera_id_end"],
            "timestamp_start": row_dict["timestamp_start"],
            "timestamp_end": row_dict["timestamp_end"],
            "speed_reading": row_dict["speed_reading"]
        }

        # Perform a stateful Upsert (Update/Insert) under a 
        # single atomic network call
        self.db[self.collection_name].update_one(
            {
                "car_plate": car_plate,
                "violation_day": violation_day
            },
            {
                "$setOnInsert": {
                    "registration_status": "flagged",
                },
                # Maximum speed reading of the violations group
                "$max": {
                    "max_speed_recorded": row_dict["speed_reading"]
                },
                # Minimum speed reading of the violations group
                "$min": {
                    "min_speed_recorded": row_dict["speed_reading"]
                },
                # The number of violations in a single group
                "$inc": {
                    "total_violations_today": 1
                },
                # The set of violations within the group
                "$addToSet": {
                    "violations_today": event_entry
                }
            },
            upsert=True
        )
    
    # Called whern all rows have been processed
    def close(self, err):
        if err:
            print(f"!!!!!\nPartition failed with error:\n{err}\n!!!!!")

        if hasattr(self, 'mongo_client'):
            self.mongo_client.close()

The violation writer attaches the DbWriter object (database writer) to append each entry in the violation database into the collection called "violations_daily_summary" of the MongoDB database a2_db. It also uses a checkpoint directory to ensure fault-tolerant tracking state recovery. This is one of the target sinks for the structured streaming process.

In [13]:
violations_writer = (
    violations_df
    .writeStream
    .outputMode("append")
    .foreach(
        DbWriter(
            db_name="a2_db",
            collection_name="violations_daily_summary"
        )
    )
    .option(
        "checkpointLocation",
        "./checkpoints/daily_violations"
    )
)

This writer tracks incomplete vehicle journeys and like the violation writer, also uses the checkpoint directory for fault-tolerant metadata tracking. This is another of the target sinks for the structured streaming process.

In [14]:
incomplete_writer = (
    incomplete_journeys
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "./checkpoints/incomplete")
)

Configure the Spark engine to bypass strict structural state validations across consecutive stream-stream joins. This is by disabling the stateful operator correctness checks.

In [15]:
# Configure spark to disable stateful operator correctness checks
spark.conf.set(
    "spark.sql.streaming.statefulOperator.checkCorrectness.enabled",
    "false"
)

# Start the lifecycle of the violations writer
writer = violations_writer
violations_query = violations_writer.start()
# Start the lifecycle of the writer for incomplete journeys
incomplete_query = incomplete_writer.start()

The block continuously awaits stream updates, intercept manual allocation stops and ends the lifecycles of the writers to prevent thread leakage.

In [16]:

try:
    violations_query.awaitTermination()
except KeyboardInterrupt:
    print("Interrupted by CTRL-C. Stopped query")
finally:
    violations_query.stop()
    if 'incomplete_query' in locals():
        incomplete_query.stop()
    

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


Interrupted by CTRL-C. Stopped query
